# AVCORP validation: frontier models

The 25 validation games (100 rows) of AVCORP on four frontier models, with the tactic
prompt and the model's own traces from the two previous rounds, as in
`8_eval_model_trace.py`. The prompt matches Qwen's except that the system prompt drops
`/no_think`.

| alias | API id | default reasoning |
|---|---|---|
| `gpt-5.6-sol` | `gpt-5.6-sol` | effort medium |
| `claude-opus-5.5` | `claude-opus-5-5` (skipped) | adaptive, effort medium |
| `claude-opus-5.5-neutral` | `claude-opus-5-5` | adaptive, prompt-neutral wording, effort medium |
| `gemini-3.1-pro` | `gemini-3.1-pro-preview` | thinking high |
| `deepseek-v4-pro` | `deepseek-v4-pro` | thinking on, effort high |

Results go to `results/`: `raw/<alias>.jsonl` (one line per row), `raw/<alias>.errors.log`,
`eval_predictions_<alias>.csv` and `eval_summary.csv`.

In [2]:
import importlib, sys
import pandas as pd
pd.set_option("display.width", 200)

for m in ("models", "prompt", "providers", "runner"):
    if m in sys.modules:
        importlib.reload(sys.modules[m])
import models, prompt, providers, runner

## 1. Check the keys and model ids

Reads each provider's model listing; no model is called.

In [3]:
sys.argv = ["models.py", "--check"]
models.main()

openai     135 models
anthropic  13 models
deepseek   2 models
google     61 models

alias            id                        status
gpt-5.6-sol      gpt-5.6-sol               present
claude-opus-5.5  claude-opus-5-5           present
claude-opus-5.5-neutralclaude-opus-5-5           present
gemini-3.1-pro   gemini-3.1-pro-preview    present
deepseek-v4-pro  deepseek-v4-pro           present

All four models resolve.


0

## 2. Inspect a prompt

No call is made. With an alias, the prior traces are that model's stored outputs.

In [3]:
print(prompt.GAMES[:5], "...")
print({g: [int(r["round_id"]) for r in prompt.ROWS_BY_GAME[g]] for g in prompt.GAMES[:3]})

['G015', 'G028', 'G043', 'G045', 'G051'] ...
{'G015': [1, 2, 3], 'G028': [1, 2, 3, 4], 'G043': [1, 2, 3, 4, 5]}


In [4]:
system, user = runner.show_prompt("G015", 3, alias=None)

=== SYSTEM ===
You are the Investigator in a 5-player Avalon game. Your player ID is P1 and your alignment is Good. You must reason about the following four players: P2, P3, P4, P5. Reason carefully from the evidence and produce only valid JSON.

=== USER (3,787 chars) ===
Public History
[R1] Round: 1
Leader: P2
Team: P1, P2
Votes: P1:Y P2:Y P3:Y P4:Y P5:Y
Quest 1 Outcome: FAIL
[R2] Round: 2
 Leader: P3
 Team: P3, P4, P5
 Votes: P1:N P2:Y P3:Y P4:Y P5:Y
 Quest 2 Outcome: FAIL
[R3] Round: 3
 Leader: P4
 Team: P1, P4
 Votes: P1:Y P2:Y P3:Y P4:Y P5:Y
 Quest 3 Outcome: FAIL

Discussion Log
Discussion after Quest 3:

P2: "Three fails with three different setups, and the only consistent thread is P1 being on two of them and voting yes on all of them—if you’re asking me to name evil, it’s P1 and the person who most reliably greenlit bad teams: P3."

P4: "Look at the cleanest pattern: Quest 2 failed on P3/P4/P5 and every single person except P1 approved it, so the sabotage almost surely came f

## 3. Pilot: the first game of each model

Game `G015` (3 rounds) on all four models in parallel. Check the token counts below
against each provider's usage page before the full run.

In [ ]:
runner.run_all(games=1)

In [ ]:
runner.progress()

## 4. The full run

All four models in parallel. Finished rows are skipped, so it is safe to interrupt and
re-run. Each model runs `runner.GAME_WORKERS` games at once (default 5); the rounds
of a game always run in order.

In [5]:
runner.run_all()


=== 4 models, 25 games each ===
  gpt-5.6-sol      start: 25 games, 5 at a time
  claude-opus-5.5  start: 25 games, 5 at a time
  gemini-3.1-pro   start: 25 games, 5 at a time
  deepseek-v4-pro  start: 25 games, 5 at a time
  gpt-5.6-sol      G015 R1    1/100    13.9s  in 701 out 974 (reasoning 92)
  gpt-5.6-sol      G043 R1    2/100    15.2s  in 840 out 1043 (reasoning 179)
  gpt-5.6-sol      G028 R1    3/100    15.2s  in 811 out 1112 (reasoning 159)
  claude-opus-5.5  G045 R1    1/100    16.1s  in 1379 out 1218
  claude-opus-5.5  G043 R1    2/100    16.4s  in 1331 out 1137
  gpt-5.6-sol      G051 R1    4/100    17.0s  in 868 out 1293 (reasoning 393)
  gpt-5.6-sol      G045 R1    5/100    17.4s  in 870 out 1317 (reasoning 392)
  claude-opus-5.5  G015 R1    3/100    17.8s  in 1120 out 1223
  claude-opus-5.5  G051 R1    4/100    17.9s  in 1357 out 1440
  gemini-3.1-pro   G015 R1    1/100    19.9s  in 717 out 2681
  gemini-3.1-pro   G045 R1    2/100    20.7s  in 890 out 2616
  gemini-3.

,rows,games_complete,mean_in,mean_gen,mean_reasoning,total_in,total_gen,hit_max,error_lines
model,,,,,,,,,
gpt-5.6-sol,71,15,2272,1695,546.0,161341,120326,0,10
claude-opus-5.5,65,6,2779,2012,NaN,180628,130787,0,19
gemini-3.1-pro,99,24,2539,5840,NaN,251342,578122,0,1
deepseek-v4-pro,100,25,2398,13353,12271.0,239790,1335312,1,0


### Or one model at a time

`games=` takes an int (the first n games) or a list of game ids. Set `game_workers` as `runner.run_model("gpt-5.6-sol", game_workers=10)`

In [7]:
runner.run_model("gpt-5.6-sol")

  gpt-5.6-sol      start: 25 games, 5 at a time
  gpt-5.6-sol      G191 R3   72/100    17.0s  in 3109 out 1555 (reasoning 269)
  gpt-5.6-sol      G175 R2   73/100    20.9s  in 2043 out 1847 (reasoning 638)
  gpt-5.6-sol      G149 R5   74/100    23.1s  in 3550 out 1975 (reasoning 661)
  gpt-5.6-sol      G173 R3   75/100    26.9s  in 3205 out 2228 (reasoning 886)
  gpt-5.6-sol      G203 R2   76/100    28.3s  in 2128 out 2284 (reasoning 1014)
  gpt-5.6-sol      G191 R4   77/100    16.6s  in 3513 out 1530 (reasoning 256)
  gpt-5.6-sol      G175 R3   78/100    18.8s  in 3346 out 1861 (reasoning 516)
  gpt-5.6-sol      G207 R1   79/100    16.9s  in 831 out 1447 (reasoning 409)
  gpt-5.6-sol      G191 R5   80/100    16.3s  in 3711 out 1612 (reasoning 281)
  gpt-5.6-sol      G203 R3   81/100    26.5s  in 3469 out 2305 (reasoning 1034)
  gpt-5.6-sol      G207 R2   82/100    18.2s  in 1932 out 1626 (reasoning 380)
  gpt-5.6-sol      G173 R4   83/100    37.6s  in 3689 out 3222 (reasoning 1740)
  

{'ok': 29, 'skipped': 71, 'failed': 0}

In [6]:
runner.run_model("claude-opus-5.5")

  claude-opus-5.5  start: 25 games, 5 at a time
  claude-opus-5.5  G045 R4  FAILED Refused: claude-opus-5.5: refusal (category=reasoning_extraction, explanation=This request was blocked as it seems to violate Anthropic's Terms of Service restrictions o
  claude-opus-5.5  G015 R3  FAILED Refused: claude-opus-5.5: refusal (category=reasoning_extraction, explanation=This request was blocked as it seems to violate Anthropic's Terms of Service restrictions o
  claude-opus-5.5  G051 R3  FAILED Refused: claude-opus-5.5: refusal (category=reasoning_extraction, explanation=This request was blocked as it seems to violate Anthropic's Terms of Service restrictions o
  claude-opus-5.5  G043 R3  FAILED Refused: claude-opus-5.5: refusal (category=reasoning_extraction, explanation=This request was blocked as it seems to violate Anthropic's Terms of Service restrictions o
  claude-opus-5.5  G028 R3  FAILED Refused: claude-opus-5.5: refusal (category=reasoning_extraction, explanation=This request was bl

{'ok': 0, 'skipped': 72, 'failed': 16}

In [3]:
runner.run_model("claude-opus-5.5-neutral", games=["G015", "G043", "G162"])

  claude-opus-5.5-neutral start: 3 games, 5 at a time
  claude-opus-5.5-neutral G162 R1    1/100    11.6s  in 1320 out 1080
  claude-opus-5.5-neutral G043 R1    2/100    17.1s  in 1320 out 1714 (reasoning 229)
  claude-opus-5.5-neutral G015 R1    3/100    18.1s  in 1109 out 1871 (reasoning 343)
  claude-opus-5.5-neutral G162 R2    4/100    25.1s  in 2573 out 2335 (reasoning 960)
  claude-opus-5.5-neutral G015 R2    5/100    24.1s  in 3078 out 2536 (reasoning 742)
  claude-opus-5.5-neutral G043 R2    6/100    26.8s  in 3042 out 2735 (reasoning 945)
  claude-opus-5.5-neutral G015 R3    7/100    17.3s  in 4912 out 1977 (reasoning 293)
  claude-opus-5.5-neutral G162 R3    8/100    31.0s  in 4177 out 3145 (reasoning 1217)
  claude-opus-5.5-neutral G043 R3    9/100    25.4s  in 4997 out 2662 (reasoning 740)
  claude-opus-5.5-neutral G043 R4   10/100    33.9s  in 5261 out 3254 (reasoning 1215)
  claude-opus-5.5-neutral G043 R5   11/100    32.6s  in 5828 out 3464 (reasoning 1362)
  claude-opus

{'ok': 11, 'skipped': 0, 'failed': 0}

In [4]:
runner.run_model("claude-opus-5.5-neutral")  # neutral wording; the run above is refused as reasoning extraction

  claude-opus-5.5-neutral start: 25 games, 5 at a time
  claude-opus-5.5-neutral G236 R2   94/100    23.5s  in 3149 out 2308 (reasoning 539)
  claude-opus-5.5-neutral G228 R2   95/100    27.5s  in 3103 out 2467 (reasoning 693)
  claude-opus-5.5-neutral G236 R3   96/100    28.3s  in 5020 out 2853 (reasoning 861)
  claude-opus-5.5-neutral G228 R3   97/100    37.1s  in 4981 out 3394 (reasoning 1422)
  claude-opus-5.5-neutral G236 R4   98/100    31.8s  in 5516 out 3302 (reasoning 1204)
  claude-opus-5.5-neutral G228 R4   99/100    23.9s  in 5536 out 2523 (reasoning 729)
  claude-opus-5.5-neutral G236 R5  100/100    40.0s  in 5756 out 3748 (reasoning 1432)
  claude-opus-5.5-neutral DONE ok=7 skipped=93 failed=0


{'ok': 7, 'skipped': 93, 'failed': 0}

In [8]:
runner.run_model("gemini-3.1-pro")

  gemini-3.1-pro   start: 25 games, 5 at a time
  gemini-3.1-pro   G236 R5  100/100    62.3s  in 3837 out 8034
  gemini-3.1-pro   DONE ok=1 skipped=99 failed=0


{'ok': 1, 'skipped': 99, 'failed': 0}

In [ ]:
runner.run_model("deepseek-v4-pro")

### A subset in parallel

```python
runner.run_all(["claude-opus-5.5", "gemini-3.1-pro"])
```

## 5. Progress and what is left

In [5]:
runner.progress()

,rows,games_complete,mean_in,mean_gen,mean_reasoning,rows_with_reasoning,total_in,total_gen,hit_max,error_lines
model,,,,,,,,,,
gpt-5.6-sol,100,25,2371,1755,589,100,237146,175524,0,10
claude-opus-5.5,72,9,2876,2036,513,1,207046,146586,0,152
claude-opus-5.5-neutral,100,25,3630,2354,629,100,362973,235446,0,12
gemini-3.1-pro,100,25,2552,5862,4608,100,255179,586156,0,1
deepseek-v4-pro,100,25,2398,13353,12271,100,239790,1335312,1,0


In [ ]:
{a: len(v) for a, v in runner.pending().items()}

## 6. Score

`allow_partial=True` scores a model's complete games only.

In [6]:
summary = runner.score_all()
summary[["run", "n", "games", "game_exact_count", "game_exact_match", "game_f1", "f1", "macro_f1",
         "parse_failures", "hit_max_new_tokens"]]

,run,n,games,game_exact_count,game_exact_match,game_f1,f1,macro_f1,parse_failures,hit_max_new_tokens
0,gpt-5.6-sol,100,25,16,0.64,0.74,0.716113,0.711667,0,0
1,claude-opus-5.5-neutral,100,25,18,0.72,0.80,0.796992,0.796667,0,0
2,gemini-3.1-pro,100,25,14,0.56,0.68,0.672043,0.658333,0,0
3,deepseek-v4-pro,100,25,12,0.48,0.62,0.624365,0.620000,1,1


### Next to the Qwen tactic runs

In [7]:
from pathlib import Path
QWEN = Path("../../trained/eval-test-model-trace/self-chain")
rows = []
for folder, label in [("tactic-grpo-h100-g16", "qwen run 6"), ("tactic-reward-grpo-h100-g16", "qwen run 7")]:
    p = QWEN / folder / "eval_summary.csv"
    if p.exists():
        for _, r in pd.read_csv(p).iterrows():
            rows.append({"run": "qwen tactic base" if r["stage"] == "base" else f"{label} {r['stage']}",
                         **{k: r[k] for k in ["n", "games", "game_exact_count", "game_exact_match", "game_f1", "f1", "macro_f1"]}})
cols = ["run", "n", "games", "game_exact_count", "game_exact_match", "game_f1", "f1", "macro_f1"]
both = pd.concat([summary[cols], pd.DataFrame(rows).drop_duplicates("run")[cols]], ignore_index=True)
both

,run,n,games,game_exact_count,game_exact_match,game_f1,f1,macro_f1
0,gpt-5.6-sol,100,25,16,0.64,0.740000,0.716113,0.711667
1,claude-opus-5.5-neutral,100,25,18,0.72,0.800000,0.796992,0.796667
2,gemini-3.1-pro,100,25,14,0.56,0.680000,0.672043,0.658333
3,deepseek-v4-pro,100,25,12,0.48,0.620000,0.624365,0.620000
4,qwen tactic base,100,25,12,0.48,0.717391,0.471380,0.372333
5,qwen run 6 sft,100,25,13,0.52,0.747475,0.599440,0.568000
6,qwen run 6 grpo,100,25,12,0.48,0.846847,0.787472,0.789667
7,qwen run 7 sft,100,25,12,0.48,0.734694,0.578035,0.541667
8,qwen run 7 grpo,100,25,18,0.72,0.882353,0.781327,0.781667
